# Python Packaging and Virtual Environments

This notebook covers how to:
- Use **virtual environments** to isolate project dependencies
- Use **pip** to install and manage packages
- Structure a Python **package**
- Use **pyproject.toml** to define a distributable package

This is essential knowledge for real-world Python development.

## 1. Virtual Environments

A **virtual environment** is an isolated Python installation for a specific project. It prevents version conflicts between projects.

### Creating a Virtual Environment
```bash
# Create
python -m venv venv

# Activate (Windows)
.\venv\Scripts\activate

# Activate (Mac/Linux)
source venv/bin/activate

# Deactivate
deactivate
```

In [ ]:
import sys
import os

# Check if we're in a virtual environment
in_venv = hasattr(sys, 'real_prefix') or (
    hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix
)

print(f'Python executable: {sys.executable}')
print(f'Python version: {sys.version.split()[0]}')
print(f'In virtual env: {in_venv}')

if in_venv:
    print(f'Venv location: {sys.prefix}')

## 2. pip — Package Installer

```bash
# Install a package
pip install requests

# Install specific version
pip install requests==2.31.0

# Install from requirements file
pip install -r requirements.txt

# Show installed packages
pip list

# Save current packages to requirements.txt
pip freeze > requirements.txt

# Uninstall
pip uninstall requests

# Show package info
pip show requests
```

In [ ]:
# List installed packages using importlib.metadata
import importlib.metadata

packages = sorted(importlib.metadata.packages_distributions())
print(f'Number of installed packages: {len(packages)}')
print('First 10 packages:')
for pkg in packages[:10]:
    try:
        version = importlib.metadata.version(pkg)
        print(f'  {pkg:20} {version}')
    except Exception:
        print(f'  {pkg}')

## 3. requirements.txt

A `requirements.txt` file lists all project dependencies. Anyone can recreate your environment with `pip install -r requirements.txt`.

In [ ]:
# Example requirements.txt content
requirements_content = """
# Core dependencies
requests==2.31.0
pandas>=2.0.0,<3.0.0
numpy>=1.24.0

# Development dependencies
pytest>=7.0.0
black>=23.0.0
mypy>=1.0.0
"""

print('Example requirements.txt:')
print(requirements_content)

# Write to file
with open('/tmp/example_requirements.txt', 'w') as f:
    f.write(requirements_content)
print('Saved to /tmp/example_requirements.txt')

## 4. Python Package Structure

A **package** is a directory with an `__init__.py` file.

### Typical Structure
```
my_package/
├── pyproject.toml          # package metadata and build config
├── README.md               # documentation
├── src/
│   └── my_package/
│       ├── __init__.py     # makes it a package; controls public API
│       ├── core.py         # main functionality
│       ├── utils.py        # helper functions
│       └── models.py       # data models
└── tests/
    ├── __init__.py
    ├── test_core.py
    └── test_utils.py
```

In [ ]:
import os

# Create a sample package structure
base = '/tmp/my_package'

dirs = [
    f'{base}/src/my_package',
    f'{base}/tests',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

# __init__.py — controls what's exported
init_content = '''
"""My Package — a demonstration package."""

__version__ = '1.0.0'
__author__ = 'Your Name'

from .core import Calculator
from .utils import format_number

__all__ = ['Calculator', 'format_number']
'''

# core.py
core_content = '''
class Calculator:
    def add(self, a, b):
        return a + b
    
    def subtract(self, a, b):
        return a - b
    
    def multiply(self, a, b):
        return a * b
    
    def divide(self, a, b):
        if b == 0:
            raise ZeroDivisionError("Cannot divide by zero")
        return a / b
'''

# utils.py
utils_content = '''
def format_number(n, decimals=2):
    return f"{n:,.{decimals}f}"
'''

files = {
    f'{base}/src/my_package/__init__.py': init_content,
    f'{base}/src/my_package/core.py': core_content,
    f'{base}/src/my_package/utils.py': utils_content,
    f'{base}/tests/__init__.py': '',
}

for path, content in files.items():
    with open(path, 'w') as f:
        f.write(content)

print('Package structure created:')
for root, dirs_list, file_list in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in file_list:
        print(f'{indent}  {f}')

## 5. `pyproject.toml` — Modern Package Configuration

`pyproject.toml` (PEP 517/518) is the modern way to configure Python packages. It replaces `setup.py`.

In [ ]:
pyproject_content = '''
[build-system]
requires = ["setuptools>=67.0", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "my-package"
version = "1.0.0"
description = "A demonstration package"
authors = [{name = "Your Name", email = "you@example.com"}]
license = {text = "MIT"}
readme = "README.md"
requires-python = ">=3.9"

dependencies = [
    "requests>=2.28.0",
    "click>=8.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.0",
    "black",
    "mypy",
]

[project.scripts]
my-tool = "my_package.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
'''

with open(f'/tmp/my_package/pyproject.toml', 'w') as f:
    f.write(pyproject_content)

print('pyproject.toml:')
print(pyproject_content)

## 6. Building and Installing Your Package

```bash
# Install in development mode (changes take effect immediately)
pip install -e .

# Build distributable files
pip install build
python -m build
# Creates:
#   dist/my_package-1.0.0.tar.gz    (source distribution)
#   dist/my_package-1.0.0-py3-none-any.whl  (wheel)

# Upload to PyPI (test)
pip install twine
twine upload --repository testpypi dist/*

# Upload to PyPI (real)
twine upload dist/*
```

## 7. The `__init__.py` — Controlling the Public API

In [ ]:
# `__all__` controls what's exported with 'from package import *'
# It also signals the intended public API.

# Example __init__.py for a package:
example = '''
__version__ = '2.0.0'

# Import key things to make them available directly
from .core import Calculator, evaluate_expression
from .models import Result, Config

# Define what's public
__all__ = [
    '__version__',
    'Calculator',
    'evaluate_expression',
    'Result',
    'Config',
]
'''

print('Example __init__.py:')
print(example)

## Common Mistakes

### Mistake 1: Installing Packages Globally Instead of in a Venv

In [ ]:
# Check if pip is pointing to a venv or global
import subprocess
import sys

result = subprocess.run([sys.executable, '-m', 'pip', '--version'], 
                        capture_output=True, text=True)
print('pip version info:')
print(result.stdout)

print('''
Best practice:
1. Always create a venv per project
2. Activate it before installing
3. Add venv/ to .gitignore
4. Commit requirements.txt to version control
''')

## Mini-Project: Package a Utility Library

In [ ]:
import os

# Create a complete mini package
pkg_dir = '/tmp/texttools'

structure = {
    f'{pkg_dir}/src/texttools/__init__.py': '''
"""TextTools — simple text processing utilities."""
__version__ = '1.0.0'
from .analysis import word_count, char_frequency, most_common
from .transform import reverse, to_title_case, truncate
__all__ = ['word_count', 'char_frequency', 'most_common', 'reverse', 'to_title_case', 'truncate']
''',
    f'{pkg_dir}/src/texttools/analysis.py': '''
from collections import Counter

def word_count(text: str) -> dict:
    words = text.lower().split()
    return {word: words.count(word) for word in set(words)}

def char_frequency(text: str) -> dict:
    return dict(Counter(text.lower()))

def most_common(text: str, n: int = 5) -> list:
    words = text.lower().split()
    return Counter(words).most_common(n)
''',
    f'{pkg_dir}/src/texttools/transform.py': '''
def reverse(text: str) -> str:
    return text[::-1]

def to_title_case(text: str) -> str:
    return text.title()

def truncate(text: str, max_len: int, suffix: str = "...") -> str:
    if len(text) <= max_len:
        return text
    return text[:max_len - len(suffix)] + suffix
''',
    f'{pkg_dir}/tests/test_analysis.py': '''
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "../src"))
from texttools import word_count, most_common

def test_word_count():
    result = word_count("the cat sat on the mat")
    assert result["the"] == 2
    assert result["cat"] == 1
    print("test_word_count: PASS")

def test_most_common():
    result = most_common("a a a b b c", n=2)
    assert result[0] == ("a", 3)
    print("test_most_common: PASS")

if __name__ == "__main__":
    test_word_count()
    test_most_common()
''',
}

for path, content in structure.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write(content)

# Use the package directly
import sys
sys.path.insert(0, f'{pkg_dir}/src')

from texttools import word_count, most_common, reverse, truncate

text = 'Python is great Python is powerful Python is fun'
print('Word count:', word_count(text))
print('Most common:', most_common(text, 3))
print('Reversed:', reverse('hello'))
print('Truncated:', truncate('This is a very long sentence', 15))